In [1]:
!git clone https://github.com/pkuliyi2015/GeoBloom.git
%cd GeoBloom
!conda install -y -c conda-forge cuda-cudart
!pip install jieba_fast xxhash==3.4.1 scikit-learn==1.4.2
!g++ nnue/v19/nnue.cpp -o nnue/v19/nnue -pthread -mavx2 -O3 -fno-tree-vectorize

Cloning into 'GeoBloom'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 313 (delta 15), reused 20 (delta 6), pack-reused 276 (from 1)
Receiving objects: 100% (313/313), 196.66 MiB | 31.04 MiB/s, done.
Resolving deltas: 100% (145/145), done.
/kaggle/working/GeoBloom
/bin/bash: line 1: conda: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 78.8 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 67.2 MB/s eta 0:00:00:00:0100:01
  Created wheel for jieba_fast: filename=jieba_fast-0.53-cp312-cp312-linux_x86_64.whl size=7659511 sha256=8518c927596479b9547a4ec9056e588cafb5368fd8e9bbe97e923497f8c5488c
  Stored in directory: /root/.cache/pip/wheels/65/67/37/8968a5b150cd26683fd229f2987694f1ff76c035f9ddc84bfe
Succes

In [2]:
import torch
import subprocess

# Kiểm tra CUDA
print("=== Kiểm tra CUDA ===")
print("CUDA có sẵn để training không?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dòng GPU đang sử dụng:", torch.cuda.get_device_name(0))

# Kiểm tra Conda
print("\n=== Kiểm tra Conda ===")
try:
    conda_version = subprocess.check_output(["conda", "--version"]).decode('utf-8').strip()
    print("Thông tin Conda:", conda_version)
except FileNotFoundError:
    print("Conda KHÔNG được cài đặt (Điều này là bình thường trên Kaggle, vì hệ thống dùng pip làm trình quản lý gói mặc định).")

=== Kiểm tra CUDA ===
CUDA có sẵn để training không?: True
Dòng GPU đang sử dụng: Tesla T4

=== Kiểm tra Conda ===
Conda KHÔNG được cài đặt (Điều này là bình thường trên Kaggle, vì hệ thống dùng pip làm trình quản lý gói mặc định).


In [3]:
!apt-get update && apt-get install -y p7zip-full
!cd data && 7z x GeoGLUE_clean.7z -y
!python model/dataset.py --dataset GeoGLUE_clean

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:13 https:

In [4]:
import re

file_path = 'model/geobloom_v19.py'
with open(file_path, 'r') as f:
    content = f.read()

# Tìm và thay thế tất cả các biến num_workers thành 0
new_content = re.sub(r'num_workers\s*=\s*\d+', 'num_workers=0', content)

with open(file_path, 'w') as f:
    f.write(new_content)
print("Đã ép num_workers về 0 để tiết kiệm RAM.")

Đã ép num_workers về 0 để tiết kiệm RAM.


In [5]:
!grep "num_workers" model/geobloom_v19.py

            train_dataloader.append(DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, collate_fn=train_dataset.collate_fn))


In [ ]:
!python model/geobloom_v19.py --dataset GeoGLUE_clean --epochs 5

In [ ]:
import time
import subprocess
import os

print("--- Bắt đầu chạy và lưu kết quả NNUE Engine tại thư mục gốc của Repo ---")
start_time = time.time()

# 1. Chạy và bắt trọn log in ra của file thực thi C++
cmd = "nnue/v19/nnue GeoGLUE_clean test 8 800-800-800-800"
process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

output_lines = []
for line in process.stdout:
    print(line, end="") # Vẫn in ra màn hình để theo dõi tiến độ thời gian thực trên Kaggle
    output_lines.append(line)

process.wait()
end_time = time.time()

# 2. Tính toán tổng thời gian và QPS ở cấp độ Python để đồng bộ dữ liệu
total_time = end_time - start_time
num_queries = 12157
qps = num_queries / total_time

# 3. Đo dung lượng đĩa của file cấu trúc cây index (.bin) của GeoBloom
tree_size_mb = 0.0
tree_path = 'data_bin/GeoGLUE_clean/nnue_v19.bin'
if os.path.exists(tree_path):
    tree_size_mb = os.path.getsize(tree_path) / (1024 * 1024)

# 4. Xác định đường dẫn lưu file ngay tại thư mục gốc của repo (cùng cấp với các folder nnue, data_bin,...)
log_file_path = 'result.txt' 

with open(log_file_path, 'w', encoding='utf-8') as f:
    f.writelines(output_lines)
    # Ghi thêm các dòng thông số đồng bộ cấu trúc với baseline
    f.write(f"\nInference_Time_Seconds: {total_time:.2f}\n")
    f.write(f"Inference_QPS: {qps:.3f}\n")
    f.write(f"Disk_Usage_MB: {tree_size_mb:.2f}\n")

print(f"\n🎉 Hệ thống đã lưu toàn bộ kết quả thành công vào file: {os.path.abspath(log_file_path)}")